# 03 — Calibration des scores

**Rôle** : transformer les scores bruts en **probabilités fiables** (quand le système annonce 70 %, il a raison environ 7 fois sur 10), et fixer les points de fonctionnement.

**Entrées** : `raw_scores.npz`, `splits.json`. **Sorties** : `calibration.json` (chargé par l'application), `operating_points.json`, diagrammes de fiabilité.

Règle d'or : **on ajuste sur le split `calibration`, on mesure sur le split `test`.**

> Une cible sans étiquette dans le dataset (ex. *Lung Opacity* avec NIH) n'est pas calibrée. L'application l'affichera alors comme « score brut, non calibré » : c'est voulu (on n'affiche jamais un pourcentage non calibré).

In [ ]:
# --- Initialisation (identique dans tous les notebooks) ---
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
IN_KAGGLE = Path("/kaggle").exists()
if IN_KAGGLE:
    # Internet doit être ACTIVÉ dans les réglages du notebook Kaggle.
    # Remplacez VOTRE_COMPTE par votre identifiant GitHub (dépôt medicure-ai).
    %pip install -q git+https://github.com/VOTRE_COMPTE/medicure-ai.git torchxrayvision
else:
    sys.path.insert(0, str(Path.cwd().parent))  # notebook lancé depuis le dossier notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from medguard.config import TARGETS, MODEL_WEIGHTS, Settings, load_settings, save_settings
from medguard.paths import find_file

# --- Configuration (à adapter) ---
DATASET = os.environ.get("MEDGUARD_DATASET", "nih")            # "nih" ou "chexpert"
DATA_ROOT = Path(os.environ.get("MEDGUARD_DATA", "/kaggle/input/data"))  # À ADAPTER au dataset attaché
WEIGHTS = os.environ.get("MEDGUARD_WEIGHTS", MODEL_WEIGHTS)
N_MAX = int(os.environ.get("MEDGUARD_NMAX", 6000))               # nb max d'images chargées
SEED = 42

WORK = Path("/kaggle/working") if IN_KAGGLE else Path(os.environ.get("MEDGUARD_WORK", "../outputs")).resolve()
ARTIFACTS = WORK / "artifacts"
FIGURES = WORK / "figures"
for d in (ARTIFACTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)
print("Kaggle :", IN_KAGGLE, "| dataset :", DATASET, "| poids :", WEIGHTS, "| sortie :", WORK)

In [ ]:
from medguard.calibration import Calibrator, expected_calibration_error, brier_score, reliability_curve
from medguard.data import load_dataset
from medguard.evaluation import bootstrap_ci, sensitivity_specificity, youden_threshold, threshold_for_sensitivity
import json

df, EVAL_TARGETS = load_dataset(DATASET, DATA_ROOT, n_max=N_MAX, seed=SEED)
labels = df.set_index(df.image_id.astype(str))

data = np.load(find_file("raw_scores.npz", [ARTIFACTS, WORK]))
ids, split, scores, names = data["ids"], data["split"], data["scores"], list(data["targets"])
cal_mask, test_mask = split == "calibration", split == "test"
print("images calibration :", cal_mask.sum(), "| test :", test_mask.sum())

def labels_for(t):
    return labels.loc[ids, t].values.astype(float)

for t in EVAL_TARGETS:
    y = labels_for(t)
    n_pos = int(np.nansum(y[cal_mask]))
    print(f"{t:14s} positifs en calibration : {n_pos}", "  (trop peu : < 30)" if n_pos < 30 else "")

## 1. Comparer les méthodes

In [ ]:
def fit_inputs(t):
    y = labels_for(t); j = names.index(t)
    ok = cal_mask & ~np.isnan(y)
    return scores[ok, j], y[ok]

def evaluate(cal):
    out = {}
    for t in EVAL_TARGETS:
        y = labels_for(t); j = names.index(t)
        ok = test_mask & ~np.isnan(y)
        p = np.asarray(cal.calibrate(t, scores[ok, j]))
        out[t] = (expected_calibration_error(p, y[ok]), brier_score(p, y[ok]))
    return out

rows = []
identity = Calibrator()
for label, cal in [("brut (aucune)", identity)] + [
    (m, Calibrator().fit({t: fit_inputs(t)[0] for t in EVAL_TARGETS}, {t: fit_inputs(t)[1] for t in EVAL_TARGETS}, method=m))
    for m in ("temperature", "platt")]:
    for t, (ece, br) in evaluate(cal).items():
        rows.append({"méthode": label, "cible": t, "ECE": ece, "Brier": br})
comparison = pd.DataFrame(rows).pivot(index="cible", columns="méthode", values="ECE").round(4)
display(comparison)

Le score brut de TorchXRayVision est normalisé autour d'un point de fonctionnement : son ECE est généralement mauvais, et la correction de Platt (pente + décalage) est la plus adaptée quand la prévalence du dataset diffère de celle de l'entraînement.

In [ ]:
METHOD = "platt"   # "temperature" ou "platt"
cal = Calibrator(fitted_on=f"{DATASET}, split calibration, poids {WEIGHTS}, méthode {METHOD}").fit(
    {t: fit_inputs(t)[0] for t in EVAL_TARGETS}, {t: fit_inputs(t)[1] for t in EVAL_TARGETS}, method=METHOD)
print("Écrit :", cal.save(ARTIFACTS / "calibration.json"))
{t: (round(v.a, 3), round(v.b, 3)) for t, v in cal.per_target.items()}

## 2. Diagrammes de fiabilité (avant / après, sur le test)

In [ ]:
fig, axes = plt.subplots(2, len(EVAL_TARGETS), figsize=(4.2 * len(EVAL_TARGETS), 7))
axes = np.array(axes).reshape(2, -1)
for c, t in enumerate(EVAL_TARGETS):
    y = labels_for(t); j = names.index(t); ok = test_mask & ~np.isnan(y)
    for r, (title, p) in enumerate([("brut", scores[ok, j]), ("calibré", np.asarray(cal.calibrate(t, scores[ok, j])))]):
        curve = reliability_curve(p, y[ok], 10); ax = axes[r, c]
        ax.plot([0, 1], [0, 1], "--", color="grey")
        ax.plot(curve["mean_predicted"], curve["fraction_positive"], "o-", color="#eb3b5a" if r == 0 else "#20bf6b")
        ax.set_title(f"{TARGETS.get(t, t)} — {title}\nECE {expected_calibration_error(p, y[ok]):.3f}")
        ax.set_xlabel("probabilité prédite"); ax.set_ylabel("fréquence observée"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout(); plt.savefig(FIGURES / "03_reliability_diagrams.png", dpi=150); plt.show()

## 3. Points de fonctionnement

Seuil de Youden (équilibre sensibilité/spécificité) et seuil garantissant 90 % de sensibilité, **choisis sur le split calibration**, **rapportés sur le split test** avec intervalles de confiance.

In [ ]:
ops, rows = {}, []
for t in EVAL_TARGETS:
    y = labels_for(t); j = names.index(t)
    okc, okt = cal_mask & ~np.isnan(y), test_mask & ~np.isnan(y)
    pc = np.asarray(cal.calibrate(t, scores[okc, j])); pt = np.asarray(cal.calibrate(t, scores[okt, j]))
    thr = {"youden": youden_threshold(y[okc], pc), "sens90": threshold_for_sensitivity(y[okc], pc, 0.90)}
    ops[t] = thr
    for name, th in thr.items():
        sens = bootstrap_ci(lambda a, b: sensitivity_specificity(a, b, th)[0], y[okt], pt, n_boot=500)
        spec = bootstrap_ci(lambda a, b: sensitivity_specificity(a, b, th)[1], y[okt], pt, n_boot=500)
        rows.append({"cible": t, "seuil": name, "valeur": round(th, 3),
                     "sensibilité": f"{sens[0]:.2f} [{sens[1]:.2f}–{sens[2]:.2f}]",
                     "spécificité": f"{spec[0]:.2f} [{spec[1]:.2f}–{spec[2]:.2f}]"})
(ARTIFACTS / "operating_points.json").write_text(json.dumps(ops, indent=2))
display(pd.DataFrame(rows))
print("Écrit :", ARTIFACTS / "operating_points.json")

### Suite
`04_risk_coverage.ipynb` : la preuve que le refus de conclure améliore la fiabilité.